
# <span style="color:blue">CDC (Change Data Capture) and Debezium</span>

# What is CDC?

CDC stands for:

```text
Change Data Capture
```

CDC is a technique used to capture database changes and propagate them to downstream systems in near real time.

### Simple Definition

```text
CDC
=
Capture Database Changes
```

Instead of repeatedly reading the entire table, CDC captures only changed records.

---

# Why Do We Need CDC?

Traditional data pipelines often use:

```sql
SELECT * FROM customer
```

at regular intervals.

---

## Problems With Full Table Reads

❌ Expensive

❌ Slow

❌ High Database Load

❌ Unnecessary Data Movement

❌ Poor Scalability

---

## CDC Solution

Instead of reading the full table:

```text
Only Changed Records
```

are captured and processed.

---

# Full Load vs CDC

## Traditional Full Load

```text
Customer Table

100 Million Records
```

Every run:

```text
Read 100 Million Records
```

even if only:

```text
10 Records Changed
```

---

## CDC

Only:

```text
10 Changed Records
```

are captured.

---

## Benefits

✅ Less Data Movement

✅ Faster Processing

✅ Near Real-Time Updates

✅ Lower Database Load

---

# CDC Architecture

```text
Database
    ↓

 Capture Changes

    ↓

    Kafka

    ↓

Consumers
```

---

# What is Debezium?

Debezium is an open-source CDC platform built on top of Kafka Connect.

### Simple Definition

```text
Debezium
=
Kafka Connect Source Connector
For CDC
```

Debezium captures database changes and streams them into Kafka.

---

# What Changes Can Debezium Capture?

✅ INSERT

✅ UPDATE

✅ DELETE

✅ INITIAL SNAPSHOT

---

# Debezium Architecture

```text
MySQL
   ↓

Debezium

   ↓

Kafka

   ↓

Consumers
```

---

# Real Architecture

```text
MySQL
   ↓

Debezium Source Connector

   ↓

Kafka

   ↓

Consumers
```

---

# How Debezium Works

A common misconception:

```text
Debezium Continuously Executes:

SELECT * FROM customer
```

❌ Incorrect

---

## Actual Working

Debezium reads:

```text
Database Transaction Logs
```

instead of querying entire tables repeatedly.

---

# Database Log Files

Different databases use different transaction logs.

---

## MySQL

```text
Binlog
```

---

## PostgreSQL

```text
WAL
```

(WAL = Write Ahead Log)

---

## Oracle

```text
Redo Log
```

---

## SQL Server

```text
Transaction Log
```

---

# Database Log Architecture

```text
Database Changes

      ↓

Transaction Log

(Binlog / WAL / Redo Log)

      ↓

Debezium

      ↓

Kafka
```

---

# Example

Suppose Customer Table contains:

```text
ID   Name

1    Arif
```

Now update:

```text
ID   Name

1    Mohammad Arif
```

---

Database writes this change into:

```text
Binlog
```

Debezium reads the Binlog and produces a Kafka event.

---

# Debezium Event Structure ⭐⭐⭐

Debezium emits structured change events.

Example:

```json
{
  "before": {
    "id": 1,
    "name": "Arif"
  },
  "after": {
    "id": 1,
    "name": "Mohammad Arif"
  },
  "op": "u"
}
```

---

# Event Fields

## before

Data before the change.

Example:

```json
{
  "id": 1,
  "name": "Arif"
}
```

---

## after

Data after the change.

Example:

```json
{
  "id": 1,
  "name": "Mohammad Arif"
}
```

---

## op

Operation type.

Example:

```text
u
```

meaning:

```text
Update
```

---

# Debezium Operation Types ⭐⭐⭐

## c

```text
Create
```

Insert operation.

---

## u

```text
Update
```

Update operation.

---

## d

```text
Delete
```

Delete operation.

---

## r

```text
Read
```

Snapshot record.

---

# Operations Summary

```text
c = Create

u = Update

d = Delete

r = Snapshot Read
```

---

# Example: INSERT Event

Database:

```text
INSERT Customer
```

Event:

```json
{
  "after": {
    "id": 1,
    "name": "Arif"
  },
  "op": "c"
}
```

---

# Example: UPDATE Event

```json
{
  "before": {
    "id": 1,
    "name": "Arif"
  },
  "after": {
    "id": 1,
    "name": "Mohammad Arif"
  },
  "op": "u"
}
```

---

# Example: DELETE Event

```json
{
  "before": {
    "id": 1,
    "name": "Arif"
  },
  "after": null,
  "op": "d"
}
```

---

# What is a Snapshot?

When Debezium starts for the first time, it captures the current database state.

This process is called:

```text
Snapshot
```

---

## Example

Customer Table:

```text
ID   Name

1    Arif

2    Ali

3    John
```

---

Debezium reads:

```text
All Existing Records
```

and publishes them.

---

# Snapshot Architecture

```text
Database

Current Data

      ↓

Snapshot

      ↓

Kafka
```

---

# Why Snapshot?

A new CDC system needs an initial view of existing data.

Snapshot provides:

```text
Initial State Of Database
```

---

# Snapshot vs Streaming ⭐⭐⭐

## Snapshot

Captures:

```text
Existing Data
```

---

Example:

```text
Current Customer Table
```

---

## Streaming

Captures:

```text
Future Changes
```

---

Example:

```text
New Inserts

Updates

Deletes
```

---

## Comparison

### Snapshot

```text
Historical State
```

---

### Streaming

```text
Incremental Changes
```

---

# Debezium Flow

```text
Snapshot

      ↓

Streaming Changes
```

This is the standard Debezium lifecycle.

---

# Tombstone Events ⭐⭐⭐

You already learned:

```text
Tombstone Records
```

during Log Compaction.

---

# Delete Flow

Customer:

```text
ID = 1
```

gets deleted.

Debezium emits:

```json
{
  "before": {
    "id": 1
  },
  "after": null,
  "op": "d"
}
```

---

Later Debezium can emit:

```text
Key = 1

Value = null
```

This becomes:

```text
Tombstone Record
```

for compacted topics.

---

# Why Tombstone Records?

They allow Kafka Log Compaction to completely remove deleted records.

---

# Real CDC Pipeline ⭐⭐⭐⭐

One of the most common Data Engineering architectures:

```text
MySQL
   ↓

Debezium Source Connector

   ↓

Kafka

   ↓

Snowflake Sink Connector

   ↓

Snowflake
```

---

# End-To-End Flow

```text
Customer Updated

      ↓

MySQL Binlog

      ↓

Debezium

      ↓

Kafka Topic

      ↓

Snowflake
```

---

# Relationship With Previous Topics

```text
Database
     ↓

Debezium

     ↓

Kafka Connect

     ↓

Avro

     ↓

Schema Registry

     ↓

Kafka

     ↓

Consumers
```

Everything now connects together.

---

# Benefits of CDC

✅ Near Real-Time Data Movement

✅ Reduced Database Load

✅ Incremental Processing

✅ Less Data Movement

✅ Better Scalability

✅ Event-Driven Architecture

✅ Faster Analytics

✅ Efficient Replication

---

# Benefits of Debezium

✅ Open Source

✅ Kafka Native

✅ Uses Database Logs

✅ Supports Multiple Databases

✅ Reliable CDC

✅ Scalable

✅ Integrates With Kafka Connect

---

# Quick Revision

```text
CDC
=
Capture Database Changes
```

---

```text
Debezium
=
Kafka Connect Source Connector
For CDC
```

---

```text
MySQL
=
Binlog
```

---

```text
PostgreSQL
=
WAL
```

---

```text
Oracle
=
Redo Log
```

---

```text
c
=
Create
```

---

```text
u
=
Update
```

---

```text
d
=
Delete
```

---

```text
r
=
Snapshot Read
```

---

```text
Snapshot
=
Existing Data
```

---

```text
Streaming
=
Future Changes
```

---

```text
Delete
+
Compaction
=
Tombstone Record
```

---

# Most Important Interview Statement ⭐

```text
Debezium is a Kafka Connect source connector that implements Change Data Capture (CDC) by reading database transaction logs and streaming INSERT, UPDATE, DELETE, and snapshot events into Kafka.
```

---

# <span style="color:red">CDC & Debezium Interview Questions and Answers</span>

## Q1. What is CDC?

CDC (Change Data Capture) is a technique used to capture INSERT, UPDATE, and DELETE operations from a database and propagate them to downstream systems.

---

## Q2. Why Do We Need CDC?

To avoid repeatedly reading full tables and process only changed records.

---

## Q3. What Problems Does CDC Solve?

✅ High Database Load

✅ Slow Batch Processing

✅ Unnecessary Data Movement

---

## Q4. What is Debezium?

Debezium is an open-source CDC platform and Kafka Connect source connector used to capture database changes.

---

## Q5. Is Debezium a Source or Sink Connector?

```text
Source Connector
```

---

## Q6. How Does Debezium Work?

Debezium reads database transaction logs and publishes change events to Kafka.

---

## Q7. Does Debezium Continuously Query Tables?

❌ No

It primarily reads transaction logs.

---

## Q8. What Does Debezium Read in MySQL?

```text
Binlog
```

---

## Q9. What Does Debezium Read in PostgreSQL?

```text
WAL
```

---

## Q10. What Does Debezium Read in Oracle?

```text
Redo Log
```

---

## Q11. What is a Snapshot?

A snapshot is the initial capture of existing data from a database.

---

## Q12. Why Is a Snapshot Needed?

To obtain the initial state of the database before streaming changes.

---

## Q13. What Happens After Snapshot Completion?

Debezium switches to streaming mode.

---

## Q14. What is Streaming Mode?

Capturing future database changes in real time.

---

## Q15. What Does the "before" Field Represent?

The record before a change occurred.

---

## Q16. What Does the "after" Field Represent?

The record after a change occurred.

---

## Q17. What Does the "op" Field Represent?

The operation type.

---

## Q18. What Does op=c Mean?

```text
Create
```

---

## Q19. What Does op=u Mean?

```text
Update
```

---

## Q20. What Does op=d Mean?

```text
Delete
```

---

## Q21. What Does op=r Mean?

```text
Snapshot Read
```

---

## Q22. What Happens When a Row is Deleted?

Debezium emits a delete event and may emit a tombstone record for compacted topics.

---

## Q23. What is a Tombstone Record in CDC?

A record with:

```text
Valid Key

Value = null
```

used for deletion in compacted topics.

---

## Q24. What Is a Common CDC Architecture?

```text
MySQL
   ↓
Debezium
   ↓
Kafka
   ↓
Consumers
```

---

## Q25. What Is a Common Data Engineering Architecture?

```text
MySQL
   ↓
Debezium Source Connector
   ↓
Kafka
   ↓
Snowflake Sink Connector
   ↓
Snowflake
```

---

## Q26. What Is the Main Benefit of Debezium?

Real-time CDC without custom code.

---

## Q27. Why Is Debezium Popular?

Because it provides reliable CDC using database transaction logs and integrates seamlessly with Kafka.

---

## Q28. Which Component Executes Debezium?

```text
Kafka Connect
```

---

## Q29. What Is the Difference Between Snapshot and Streaming?

```text
Snapshot
=
Existing Data
```

```text
Streaming
=
Future Changes
```

---

## Q30. Most Important CDC Interview Answer?

```text
Debezium is a Kafka Connect source connector that implements CDC by reading database transaction logs and streaming database changes into Kafka in near real time.
```


# <span style="color:blue">How CDC and Debezium Actually Work?</span>

# Simple Problem Statement

Suppose we have a MySQL table:

```text
Customer

ID   Name

1    Arif
```

Now a user updates the record:

```text
Customer

ID   Name

1    Mohammad Arif
```

Question:

```text
How does Kafka know that the data changed?
```

This is exactly where:

```text
CDC (Change Data Capture)
```

comes into the picture.

---

# Without CDC

Many traditional systems refresh data by repeatedly reading entire tables.

Example:

```sql
SELECT * FROM Customer;
```

every few minutes.

---

## Problem

Suppose:

```text
Customer Table
=
1 Million Rows
```

Only:

```text
1 Row Changed
```

but the application still reads:

```text
1 Million Rows
```

every time.

Problems:

❌ Slow

❌ Expensive

❌ High Database Load

❌ Unnecessary Data Movement

---

# With CDC

Instead of reading the full table again and again:

```text
Only Changed Records
```

are captured.

---

# How Does a Database Know What Changed?

Databases maintain transaction logs.

For MySQL:

```text
Binlog
```

---

# What is Binlog?

Think of Binlog as a diary maintained by MySQL.

Whenever something changes, MySQL writes an entry into the Binlog.

---

## Insert Example

Database:

```text
INSERT Customer 1
```

Binlog Entry:

```text
Customer Added
```

---

## Update Example

Database:

```text
Arif
   ↓
Mohammad Arif
```

Binlog Entry:

```text
Customer Updated
```

---

## Delete Example

Database:

```text
Delete Customer 1
```

Binlog Entry:

```text
Customer Deleted
```

---

# What is CDC?

CDC means:

```text
Capturing Database Changes
```

Specifically:

```text
INSERT

UPDATE

DELETE
```

operations.

---

# What Does Debezium Do?

A common misconception:

Debezium continuously executes:

```sql
SELECT * FROM Customer;
```

❌ Wrong

Debezium does NOT continuously scan database tables.

---

## Actual Role of Debezium

Debezium continuously monitors:

```text
Database Transaction Logs
```

For MySQL:

```text
Binlog
```

---

# Actual Architecture

```text
MySQL
   ↓

Binlog

   ↓

Debezium

   ↓

Kafka
```

---

# Real Flow Step-by-Step

Suppose customer data is updated.

Before:

```text
ID   Name

1    Arif
```

After:

```text
ID   Name

1    Mohammad Arif
```

---

## Step 1

User updates the record.

```text
Customer Table Updated
```

---

## Step 2

MySQL writes the change into:

```text
Binlog
```

Example:

```text
Customer ID=1 Updated
```

---

## Step 3

Debezium is continuously listening to the Binlog.

It detects:

```text
Customer Updated
```

---

## Step 4

Debezium creates a Kafka Change Event.

Example:

```json
{
  "before": {
    "id": 1,
    "name": "Arif"
  },
  "after": {
    "id": 1,
    "name": "Mohammad Arif"
  },
  "op": "u"
}
```

---

## Step 5

Debezium publishes this event into Kafka.

```text
Kafka Topic
```

---

## Step 6

Consumers receive the update.

Examples:

```text
Snowflake

Elasticsearch

Analytics Service

Data Warehouse
```

---

# When Does Debezium Work?

## First Time Startup

When Debezium starts for the first time:

```text
Snapshot
```

is taken.

---

### Example

Database Table:

```text
1   Arif

2   Ali

3   John
```

Debezium reads all existing records and publishes them to Kafka.

This process is called:

```text
Snapshot
```

---

# Snapshot Diagram

```text
Database

Current Data

      ↓

Snapshot

      ↓

Kafka
```

---

# After Snapshot

Debezium switches into:

```text
Streaming Mode
```

Now it continuously listens to:

```text
Binlog
```

---

## Streaming Flow

```text
INSERT
UPDATE
DELETE

      ↓

Binlog

      ↓

Debezium

      ↓

Kafka
```

Near real-time updates start flowing.

---

# Real CDC Pipeline

```text
Customer Updated

       ↓

MySQL

       ↓

Binlog

       ↓

Debezium

       ↓

Kafka Topic

       ↓

Snowflake / Consumers
```

---

# Easy Real-Life Analogy

Imagine:

```text
MySQL
=
Teacher
```

---

```text
Binlog
=
Attendance Register
```

---

```text
Debezium
=
Class Monitor
```

---

The monitor continuously watches the attendance register.

Whenever the teacher writes:

```text
New Student Added
```

the monitor immediately announces:

```text
Kafka, listen!

A new student was added.
```

The monitor does NOT check the entire classroom repeatedly.

Instead:

```text
Monitor Reads Register
```

Similarly:

```text
Debezium Reads Database Logs
```

instead of repeatedly reading database tables.

---

# Quick Revision

```text
CDC
=
Capture Database Changes
```

---

```text
Debezium
=
CDC Tool
```

---

```text
Debezium
=
Kafka Connect Source Connector
```

---

```text
Debezium Reads
=
Database Transaction Logs
```

---

```text
MySQL Log
=
Binlog
```

---

```text
First Run
=
Snapshot
```

---

```text
After Snapshot
=
Streaming Mode
```

---

# Most Important Interview Answer ⭐

```text
CDC captures database changes such as INSERT, UPDATE, and DELETE.

Debezium implements CDC by continuously reading database transaction logs (such as MySQL Binlog) and publishing those changes as events to Kafka in near real time.
```

---

# Memory Trick

```text
Database Change
       ↓
Binlog
       ↓
Debezium
       ↓
Kafka
       ↓
Consumer
```

This is the complete end-to-end CDC flow.